# 🎮 Minecraft AI Builder - Training Notebook

Generate professional-quality Minecraft builds with AI!

## 📋 Two Training Options:

### Option A: BASIC (Faster) - 8-12 hours
- Good quality
- Text-conditioned generation

### Option B: HIGH-QUALITY (Best) ⭐ RECOMMENDED - 20-28 hours  
- Excellent quality (0.85+ score)
- Diffusion model
- Validation & auto-fix

**Choose your pipeline below!**

## 🚀 Setup

In [ ]:
# Clone repository
!git clone https://github.com/GogaGogich123/Ai.git
%cd Ai
!git checkout capy/cap-1-7fa451e9

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 🧪 Test Suite

In [ ]:
!python test_training.py

## 📦 Login to Weights & Biases (Optional)

Track training metrics at [wandb.ai](https://wandb.ai)

In [ ]:
import wandb
wandb.login()

---
# 🎯 OPTION A: BASIC PIPELINE (Faster)

**Training time:** ~8-12 hours  
**Quality:** Good  
**Features:** Text-conditioned generation

### A1. Train VQ-VAE (Stage 1) - ~4-6 hours

In [ ]:
!python mcbuilder/train_vqvae.py \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size 4 \
    --num_workers 2 \
    --embedding_dim 64 \
    --num_embeddings 512 \
    --lr 1e-4 \
    --epochs 50 \
    --save_every 5

### A2. Train Text-Conditioned Transformer (Stage 2) - ~4-6 hours

In [ ]:
!python mcbuilder/train_text_conditioned.py \
    --vqvae_checkpoint ./checkpoints/vqvae_final.pt \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_text \
    --chunk_size 32 \
    --batch_size 8 \
    --num_workers 2 \
    --mask_ratio 0.15 \
    --d_model 512 \
    --nhead 8 \
    --num_layers 12 \
    --lr 1e-4 \
    --epochs 50 \
    --save_every 5

### A3. Generate from Text Prompt

In [ ]:
!python generate_text.py \
    --vqvae_checkpoint ./checkpoints/vqvae_final.pt \
    --transformer_checkpoint ./checkpoints_text/text_transformer_final.pt \
    --prompt "a cozy medieval cottage" \
    --size 32,32,32 \
    --output cottage.litematic \
    --temperature 1.0 \
    --num_iterations 10

---
# ⭐ OPTION B: HIGH-QUALITY PIPELINE (Best)

**Training time:** ~20-28 hours  
**Quality:** Excellent (0.85+)  
**Features:** Diffusion model + Validation + Auto-fix

### B1. Train Improved VQ-VAE (Stage 1) - ~8-12 hours

Better compression with larger codebook and perceptual loss

In [ ]:
!python mcbuilder/train_improved_vqvae.py \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_improved \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size 4 \
    --num_workers 2 \
    --embedding_dim 128 \
    --num_embeddings 1024 \
    --num_res_blocks 3 \
    --lr 1e-4 \
    --epochs 100 \
    --save_every 10

### B2. Train Latent Diffusion (Stage 2) - ~12-16 hours

SOTA diffusion model for high-quality generation

In [ ]:
!python mcbuilder/train_diffusion.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --cache_dir ./data/cache \
    --checkpoint_dir ./checkpoints_diffusion \
    --chunk_size 32 \
    --overlap 4 \
    --min_blocks 800 \
    --max_blocks 50000 \
    --batch_size 4 \
    --num_workers 2 \
    --model_channels 128 \
    --num_res_blocks 2 \
    --num_heads 8 \
    --dropout 0.1 \
    --timesteps 1000 \
    --lr 1e-4 \
    --epochs 100 \
    --save_every 10

### B3. Generate High-Quality Builds with Validation

Generates multiple candidates, validates physics & interiors, selects best

In [ ]:
# Generate single high-quality build
!python generate_hq.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --diffusion_checkpoint ./checkpoints_diffusion/diffusion_final.pt \
    --size 32,32,32 \
    --output hq_castle.litematic \
    --name "AI Castle" \
    --num_samples 5 \
    --validate

In [ ]:
# Generate multiple variants
!python generate_hq.py \
    --vqvae_checkpoint ./checkpoints_improved/improved_vqvae_final.pt \
    --diffusion_checkpoint ./checkpoints_diffusion/diffusion_final.pt \
    --size 32,32,32 \
    --output hq_house.litematic \
    --name "AI House" \
    --num_samples 3 \
    --validate \
    --generate_multiple 5

---
## 📥 Download Generated Files

In [ ]:
from google.colab import files
import os

# Download all .litematic files
for filename in os.listdir('.'):
    if filename.endswith('.litematic'):
        print(f"Downloading {filename}...")
        files.download(filename)

## 💾 Download Checkpoints

Save your trained models for later use

In [ ]:
from google.colab import files
import os

# Download BASIC pipeline checkpoints
checkpoint_dirs = ['./checkpoints', './checkpoints_text']

# Or download HQ pipeline checkpoints (comment above, uncomment below)
# checkpoint_dirs = ['./checkpoints_improved', './checkpoints_diffusion']

for checkpoint_dir in checkpoint_dirs:
    if os.path.exists(checkpoint_dir):
        for filename in os.listdir(checkpoint_dir):
            if filename.endswith('.pt'):
                filepath = os.path.join(checkpoint_dir, filename)
                print(f"Downloading {filepath}...")
                files.download(filepath)

---
## 📊 Quality Metrics

Check validation results

In [ ]:
# View generated builds info
!ls -lh *.litematic

---
## 🎮 How to Use in Minecraft

1. Install [Litematica mod](https://www.curseforge.com/minecraft/mc-mods/litematica) for Forge 1.19.2
2. Download generated `.litematic` files from above
3. Place files in `.minecraft/schematics/`
4. Load in-game with Litematica menu (M key)
5. Place and paste the build!

---

## 📚 Documentation

- [QUICKSTART.md](QUICKSTART.md) - Quick reference
- [HQ_PIPELINE.md](HQ_PIPELINE.md) - High-quality pipeline details
- [QUALITY_IMPROVEMENTS.md](QUALITY_IMPROVEMENTS.md) - What makes it better

## 🐛 Troubleshooting

**Out of Memory:**
- Reduce `batch_size` (4→2 or 2→1)
- Reduce `chunk_size` (32→24)

**Slow Training:**
- Reduce `num_workers` (2→0)
- Use smaller model (basic pipeline)

**Connection Issues:**
- BuildPaste API might be slow
- Cached data is reused automatically

---

Made with ❤️ using AI